# GC-Chess_Engine — `GCNNUE05` (GC-RelGeo Dual-Stream Pure NNUE) Colab Pipeline

Notebook ini menjalankan pipeline lengkap di Google Colab:
1. Build engine C++ (`./output/main`) di Linux Colab
2. Generate dataset **Stratified Pure NNUE** (`40% mid / 40% practical endgame / 20% late endgame` + `endgame_seeds.epd`)
3. Training 2 tahap di GPU (`GCNNUE05`: `CP-HG 768->512x2` + `RelGeo 2619->128` + `MLP32` + `V * (1 - DrawGate)`)
4. Verifikasi paritas C++ ↔ Python & uji konversi endgame 100% Pure NNUE tanpa rule manual
5. Download `gc_v5.nnue` (`1,512,664` bytes)

In [ ]:
!nvidia-smi
import os
# Sesuaikan URL repo jika clone dari GitHub, atau upload zip workspace GC-Chess_Engine ke Colab:
REPO_DIR = "/content/GC-Chess_Engine"
if not os.path.isdir(REPO_DIR):
    print("Upload folder/zip GC-Chess_Engine ke /content/GC-Chess_Engine atau git clone repo Anda di sini.")
else:
    %cd {REPO_DIR}

In [ ]:
# 1. Build Linux binary ./output/main (fallback LFLAGS tanpa -static jika glibc static tidak terpasang)
!make clean && (make -j$(nproc) || make -j$(nproc) LFLAGS="-flto -pthread")
!echo "perft 5" | ./output/main

In [ ]:
# 2. Datagen via kaggle_datagen.py: Stratified (40% mid / 40% prac-eg / 20% late-eg) + Endgame seeds
!python3 tuner/kaggle_datagen.py \
    --engine ./output/main \
    --nnue gc.nnue \
    --games 9000 \
    --batch 1000 \
    --nodes 4000 \
    --eg-nodes 6000 \
    --eg-ratio 0.30 \
    --endgame-seeds tuner/endgame_seeds.epd \
    --out output/data_v5

In [ ]:
# 3. Training GCNNUE05 di GPU via train_nnue.py (tambahkan file dataset utama jika ada)
!python3 tuner/train_nnue.py \
    --arch v5 \
    --data output/data_v5/dataset_all.txt \
    --init-from gc.nnue \
    --epochs 25 \
    --batch-size 8192 \
    --lr 0.001 \
    --lambda-wdl 0.65 \
    --cp-loss-weight 0.25 \
    --device cuda \
    -o output/gc_v5.nnue

In [ ]:
# 4. Verifikasi paritas integer C++ <-> Python & uji evaluasi 100% Pure GCNNUE05
!python3 tuner/train_nnue.py --test-parity ./output/main
!python3 tuner/train_nnue.py --verify-only output/gc_v5.nnue
!printf "setoption name EvalFile value output/gc_v5.nnue\n" \
        "position fen 8/8/8/3k4/8/8/4B3/3K1N2 w - - 0 1\neval\n" \
        "position fen 8/4k3/3p4/8/2B1P3/8/4b3/4K3 w - - 72 60\neval\n" \
        "quit\n" | ./output/main

In [ ]:
# 5. Download bobot hasil training ke PC lokal
try:
    from google.colab import files
    files.download("output/gc_v5.nnue")
except ImportError:
    print("Bukan sesi Colab; file tersimpan di output/gc_v5.nnue")